In [1]:
# ==========================================
# Setup & Reproducibility
# ==========================================
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
!wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
!unzip data.zip

In [3]:
!pwd

/content


In [4]:
data_dir = "/content/data"
# basic transformations
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = train_transforms


In [5]:
# create dataloaders
train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transforms)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, "test"),  transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=20, shuffle=False)

len(train_dataset), len(test_dataset)


(800, 201)

In [6]:
class HairNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)

        # compute flatten size
        # after conv: (32, 198, 198)
        # after pool: (32, 99, 99)
        self.flatten_dim = 32 * 99 * 99

        self.fc1 = nn.Linear(self.flatten_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(self.relu(self.conv(x)))
        x = x.view(-1, self.flatten_dim)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)   # logits → BCEWithLogitsLoss handles sigmoid
        return x

model = HairNet().to(device)
model


HairNet(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=313632, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)

In [7]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)


In [9]:
from torchsummary import summary
summary(model, input_size=(3, 200, 200))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
              ReLU-2         [-1, 32, 198, 198]               0
         MaxPool2d-3           [-1, 32, 99, 99]               0
            Linear-4                   [-1, 64]      20,072,512
              ReLU-5                   [-1, 64]               0
            Linear-6                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 21.54
Params size (MB): 76.57
Estimated Total Size (MB): 98.57
----------------------------------------------------------------


In [8]:
num_epochs = 10
history = {"acc": [], "loss": [], "val_acc": [], "val_loss": []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train

    history["loss"].append(epoch_loss)
    history["acc"].append(epoch_acc)

    # validation
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)

    val_epoch_loss = val_loss / len(test_dataset)
    val_epoch_acc = correct_val / total_val

    history["val_loss"].append(val_epoch_loss)
    history["val_acc"].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs} "
          f"- Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f} "
          f"- ValLoss: {val_epoch_loss:.4f}, ValAcc: {val_epoch_acc:.4f}")


Epoch 1/10 - Loss: 0.6462, Acc: 0.6362 - ValLoss: 0.6032, ValAcc: 0.6517
Epoch 2/10 - Loss: 0.5475, Acc: 0.7100 - ValLoss: 0.7251, ValAcc: 0.6318
Epoch 3/10 - Loss: 0.5533, Acc: 0.7250 - ValLoss: 0.5991, ValAcc: 0.6716
Epoch 4/10 - Loss: 0.4802, Acc: 0.7712 - ValLoss: 0.6033, ValAcc: 0.6567
Epoch 5/10 - Loss: 0.4334, Acc: 0.8025 - ValLoss: 0.6196, ValAcc: 0.6766
Epoch 6/10 - Loss: 0.3740, Acc: 0.8325 - ValLoss: 0.7371, ValAcc: 0.6766
Epoch 7/10 - Loss: 0.2721, Acc: 0.8838 - ValLoss: 0.9223, ValAcc: 0.6418
Epoch 8/10 - Loss: 0.2478, Acc: 0.9000 - ValLoss: 0.7294, ValAcc: 0.7214
Epoch 9/10 - Loss: 0.2075, Acc: 0.9200 - ValLoss: 0.7523, ValAcc: 0.7015
Epoch 10/10 - Loss: 0.1494, Acc: 0.9450 - ValLoss: 0.7894, ValAcc: 0.7015


In [10]:
# data augmentation for next 10 epochs

augmented_train_transforms = transforms.Compose([
    transforms.RandomRotation(50),
    transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_dataset_aug = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=augmented_train_transforms)
train_loader_aug  = DataLoader(train_dataset_aug, batch_size=20, shuffle=True)


In [11]:
aug_epochs = 10
test_losses_aug = []
test_accs_aug = []

for epoch in range(aug_epochs):
    model.train()
    for images, labels in train_loader_aug:
        images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    # ---- Evaluate on test set ----
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_loss = total_loss / len(test_dataset)
    test_acc = correct / total

    test_losses_aug.append(test_loss)
    test_accs_aug.append(test_acc)

    print(f"[AUG] Epoch {epoch+11}/20 - Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")


[AUG] Epoch 11/20 - Test Loss: 0.6221, Test Acc: 0.7114
[AUG] Epoch 12/20 - Test Loss: 0.5668, Test Acc: 0.6716
[AUG] Epoch 13/20 - Test Loss: 0.5703, Test Acc: 0.7114
[AUG] Epoch 14/20 - Test Loss: 0.5339, Test Acc: 0.7114
[AUG] Epoch 15/20 - Test Loss: 0.7584, Test Acc: 0.6617
[AUG] Epoch 16/20 - Test Loss: 0.5993, Test Acc: 0.6965
[AUG] Epoch 17/20 - Test Loss: 0.6083, Test Acc: 0.7015
[AUG] Epoch 18/20 - Test Loss: 0.5586, Test Acc: 0.7065
[AUG] Epoch 19/20 - Test Loss: 0.5204, Test Acc: 0.7363
[AUG] Epoch 20/20 - Test Loss: 0.5391, Test Acc: 0.7214


In [12]:
import numpy as np

print("Median train acc:", np.median(history["acc"]))
print("Std of train loss:", np.std(history["loss"]))
print("Mean test loss with augmentation:", np.mean(test_losses_aug))
print("Avg test acc (last 5 aug epochs):", np.mean(test_accs_aug[5:]))


Median train acc: 0.8175
Std of train loss: 0.15896418235516768
Mean test loss with augmentation: 0.5877108227924921
Avg test acc (last 5 aug epochs): 0.7124378109452736
